# Baba Is AI: Reproducibility & Determinism
This notebook mimics the exact methodology used for ViZDoom to verify that replaying recorded actions produces mathematically identical `reward` and `terminated` states, proving the environment is fully deterministic when seeded.


In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

sys.path.append("..")
from fmri_gym.adapters.baba import BabaAdapter

N_REPLAYS = 100

DATASET_DIR = "../data/sub-01_20260910-155703"


### Validation Core
We define the inspection and replay loops exactly like ViZDoom, managing `episode_seeds` for dynamic restarts.


In [ ]:
def inspect_dataset(data_dir):
    print(f"--- Inspecting {os.path.basename(data_dir)} ---")
    manifest_path = os.path.join(data_dir, "manifest.json")
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    
    game_curriculum = next(p for p in manifest["curriculum"] if p.get("type") == "game")
    game_phase = next(p for p in manifest["phases"] if p.get("type") == "game")
    npz_name = game_phase["data_file"]
    npz_path = os.path.join(data_dir, npz_name)
    data = np.load(npz_path, allow_pickle=True)
    
    return manifest, data, game_curriculum, game_phase

def run_replays(game_curr, ref_data, n_replays):
    ref_actions = ref_data["actions"]
    ref_rewards = ref_data["rewards"]
    ref_terms = ref_data["terminated"]
    ref_truncs = ref_data["truncated"]
    
    ref_episode_ids = ref_data.get("episode_id", np.zeros(len(ref_actions), dtype=int))
    ref_episode_seeds = ref_data["episode_seeds"]
    
    T = len(ref_actions)
    results = []
    
    print(f"Starting {n_replays} replays of {game_curr['game']} ({T} steps each)...")
    env = None
    
    for r in range(n_replays):
        current_ep = -1
        diverged_step = None
        diverged_vars = []
        
        for t in range(T):
            ep_id = int(ref_episode_ids[t])
            
            # If the recorded step belongs to a new episode, re-seed and reset the environment
            if ep_id != current_ep:
                if env is not None: env.close()
                env = BabaAdapter(game_curr)
                seed = int(ref_episode_seeds[ep_id])
                env.reset(seed=seed)
                current_ep = ep_id
                
            obs, reward, term, trunc, info = env.step(ref_actions[t])
            
            mismatches = []
            if not np.isclose(reward, ref_rewards[t], atol=1e-5): mismatches.append("reward")
            if bool(term) != bool(ref_terms[t]): mismatches.append("terminated")
            if bool(trunc) != bool(ref_truncs[t]): mismatches.append("truncated")
            
            if len(mismatches) > 0 and diverged_step is None:
                diverged_step = t
                diverged_vars = mismatches
                break
                
        exact = (diverged_step is None)
        results.append({
            "replay_idx": r, "exact": exact, "diverged_step": diverged_step,
            "diverged_vars": diverged_vars, "reward_match": exact or ("reward" not in diverged_vars),
            "term_match": exact or ("terminated" not in diverged_vars),
            "trunc_match": exact or ("truncated" not in diverged_vars)
        })
        
    env.close()
    return pd.DataFrame(results)



### Execute Experiment


In [ ]:
manifest, data, curr, phase = inspect_dataset(DATASET_DIR)
game_name = curr["game"]

df = run_replays(curr, data, N_REPLAYS)
print(f"Dataset ({game_name}): {df['exact'].sum()} / {N_REPLAYS} fully matched.")


if df['exact'].sum() < N_REPLAYS:
    print("\n--- DIVERGENCE DETAILS ---")
    bad_runs = df[~df['exact']]
    for _, row in bad_runs.head(3).iterrows():
        print(f"Replay {row['replay_idx']} diverged at step {row['diverged_step']}")
        print(f"Variables that mismatched: {row['diverged_vars']}")
        
    print("\nLet's deeply inspect the first failure point:")
    first_bad = bad_runs.iloc[0]
    bad_step = first_bad['diverged_step']
    
    print(f"\nAt Step {bad_step}:")
    print(f"Action played: {data['actions'][bad_step]}")
    print(f"Expected Reward: {data['rewards'][bad_step]}")
    print(f"Expected Terminated: {data['terminated'][bad_step]}")


### Summary Table


In [ ]:
%matplotlib inline
total_steps = len(data["actions"])
total_states = len(df) * total_steps
perfect_runs = df['exact'].sum()
divergent_runs = len(df) - perfect_runs
match_rate = (perfect_runs / len(df)) * 100

diverged_vars_set = set()
for vlist in df['diverged_vars']:
    diverged_vars_set.update(vlist)
diverged_str = ", ".join(diverged_vars_set) if diverged_vars_set else "None"

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis("off")

table_data = [
    ["Total Replays Executed", f"{len(df)}"],
    ["Total Timesteps / Run", f"{total_steps:,}"],
    ["Total States Verified", f"{total_states:,}"],
    ["Perfect Trajectories", f"{perfect_runs}"],
    ["Divergent Trajectories", f"{divergent_runs}"],
    ["Trajectory Match Rate", f"{match_rate:.1f}%"],
    ["Diverged Variables (Flags)", diverged_str]
]

table = ax.table(cellText=table_data, colLabels=["Metric", "Result"], cellLoc="center", loc="center", colColours=["#2b5c8f", "#2b5c8f"])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", weight="bold")
    elif row > 0 and col == 1:
        metric = table_data[row-1][0]
        if "Perfect Trajectories" in metric and perfect_runs == len(df): cell.set_facecolor("#e8f5e9")
        elif "Divergent Trajectories" in metric and divergent_runs == 0: cell.set_facecolor("#e8f5e9")
        elif "Divergent Trajectories" in metric and divergent_runs > 0: cell.set_facecolor("#ffebee")
        elif "Match Rate" in metric and match_rate == 100.0: cell.set_facecolor("#e8f5e9")
        elif "Match Rate" in metric and match_rate < 100.0: cell.set_facecolor("#ffebee")
        elif "Diverged Variables" in metric:
            if diverged_str == "None": cell.set_facecolor("#e8f5e9")
            else: cell.set_facecolor("#fff3e0")

plt.title(f" Reproducibility", fontsize=14, pad=20, weight="bold")
plt.tight_layout()
plt.show()
